# UNet Cityscapes — Training on Colab

**Workflow:**
1. GPU check
2. Mount Drive + clone repo
3. Copy data from Drive → /content/ (fast SSD)
4. Install dependencies
5. Configure & train
6. Evaluate & visualize

> **Before running:** upload `cityscapes_preprocessed.zip` to your Google Drive root.

## 1 — GPU Check

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found. Change runtime to GPU in Runtime > Change runtime type."

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

# Adapt batch size to available VRAM
BATCH_SIZE = 16 if vram_gb >= 30 else 8
print(f"Batch size: {BATCH_SIZE}")

## 2 — Mount Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# TODO: replace with your GitHub repo URL
REPO_URL = "https://github.com/YOUR_USERNAME/road-segmentation.git"

import subprocess
result = subprocess.run(
    ["git", "clone", REPO_URL, "/content/road-segmentation"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

%cd /content/road-segmentation

## 3 — Install Dependencies

In [ ]:
!pip install -r requirements.txt -q
print("Dependencies installed.")

## 4 — Copy Data from Drive → /content/

> **Why?** Reading directly from Drive during training is ~10× slower than reading from /content/ SSD.
> This one-time copy takes ~30–60s and pays off across every epoch.

In [ ]:
import time

# Path to the zip on Drive — adjust if you put it in a subfolder
DRIVE_ZIP = "/content/drive/MyDrive/cityscapes_preprocessed.zip"

t0 = time.time()
!unzip -q "{DRIVE_ZIP}" -d /content/
elapsed = time.time() - t0
print(f"Data ready in {elapsed:.1f}s")

# Quick sanity checks
!echo "Train samples:" && wc -l /content/preprocessed/train.csv
!echo "Val samples  :" && wc -l /content/preprocessed/val.csv
!ls /content/preprocessed/

## 5 — Configuration

In [ ]:
import os
import yaml

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

# Override paths for Colab
cfg["data"]["root"] = "/content/preprocessed"
cfg["training"]["batch_size"] = BATCH_SIZE
cfg["checkpointing"]["dir"] = "/content/drive/MyDrive/road_seg_checkpoints"

os.makedirs(cfg["checkpointing"]["dir"], exist_ok=True)

print("Config loaded:")
print(f"  data root      : {cfg['data']['root']}")
print(f"  batch size     : {cfg['training']['batch_size']}")
print(f"  epochs         : {cfg['training']['epochs']}")
print(f"  lr             : {cfg['training']['lr']}")
print(f"  mixed precision: {cfg['training']['mixed_precision']}")
print(f"  checkpoint dir : {cfg['checkpointing']['dir']}")

## 6 — (Optional) Resume from Checkpoint

In [ ]:
import glob

RESUME = False   # Set True to resume from last checkpoint

resume_path = None
if RESUME:
    last = "/content/drive/MyDrive/road_seg_checkpoints/last.pth"
    if os.path.exists(last):
        resume_path = last
        print(f"Will resume from: {resume_path}")
    else:
        print("No checkpoint found, starting from scratch.")

## 7 — Train

In [ ]:
import sys
sys.path.insert(0, "/content/road-segmentation")

from src.train import train

model = train(cfg, resume_from=resume_path)

## 8 — Evaluate & Visualize on Validation Set

In [ ]:
import torch
import matplotlib.pyplot as plt

from src.dataset import build_loaders
from src.metrics import compute_miou, CLASS_NAMES
from src.utils import visualize_predictions, load_checkpoint
from src.model import build_model

device = "cuda"

# Load best checkpoint
import glob
best_ckpts = sorted(glob.glob("/content/drive/MyDrive/road_seg_checkpoints/best_*.pth"))
if best_ckpts:
    best_path = best_ckpts[-1]
    print(f"Loading best checkpoint: {best_path}")
    model = build_model(cfg).to(device)
    epoch, best_miou = load_checkpoint(best_path, model, device=device)
    print(f"Best val mIoU: {best_miou:.4f} (epoch {epoch+1})")
else:
    print("Using model from training session.")
    model = model.to(device)

model.eval()
_, val_loader = build_loaders(cfg)

# Per-class mIoU on full val set
all_ious = []
with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        with torch.cuda.amp.autocast():
            logits = model(images)
        preds = logits.argmax(dim=1)
        miou, per_class = compute_miou(preds, masks, num_classes=cfg["model"]["num_classes"])
        all_ious.append(per_class)

# Average per class across batches (ignoring NaN)
import numpy as np
all_ious = np.array(all_ious, dtype=float)   # (n_batches, n_classes-1)
mean_per_class = np.nanmean(all_ious, axis=0)
mean_miou = np.nanmean(mean_per_class)

print(f"\nVal mIoU: {mean_miou:.4f}")
print("Per-class IoU:")
for name, iou in zip(CLASS_NAMES[1:], mean_per_class):
    print(f"  {name:<15}: {iou:.4f}")

In [ ]:
# Visual check — first batch of val set
images, masks = next(iter(val_loader))
images, masks = images.to(device), masks.to(device)
with torch.no_grad(), torch.cuda.amp.autocast():
    logits = model(images)
preds = logits.argmax(dim=1)

fig = visualize_predictions(images, masks, preds, n=4)
plt.show()